# Docker 入門

**学習目標**
  * Docker コンテナのビルドと実行
  * Docker Hub および Google Artifact Registry からの Docker イメージのプル
  * Google Artifact Registry への Docker イメージのプッシュ

## 概要

Docker は、アプリケーションの開発、配信、実行のためのオープン プラットフォームです。Docker を使用すると、アプリケーションをインフラストラクチャから切り離し、インフラストラクチャをマネージド アプリケーションのように扱うことができます。Docker を活用することで、コードの迅速な出荷、テストの迅速化、デプロイの迅速化が可能になり、コードの記述から実行までのサイクルを短縮できます。

Docker は、カーネルのコンテナ化機能と、アプリケーションの管理およびデプロイを支援するワークフローやツールを組み合わせることでこれを実現しています。

Docker コンテナは Kubernetes で直接使用できるため、Kubernetes Engine で簡単に実行できます。Docker の基礎を習得することで、Kubernetes やコンテナ化されたアプリケーションの開発を始めるためのスキルが身につきます。

## 基本的な Docker コマンド

保有している Docker イメージを確認します。

In [ ]:
!docker images

初めて Docker を使用する場合、リポジトリは何も一覧に表示されません。

 `docker run` を使用して、パブリック レジストリから `hello-world` という Docker イメージをプルしてみましょう。Docker デーモンは `hello-world` イメージを検索し、ローカルにイメージが見つからない場合は、Docker Hub と呼ばれるパブリック レジストリからイメージをプルし、そのイメージからコンテナを作成して実行します。

In [ ]:
!docker run hello-world

Docker イメージの一覧を再度確認すると、`hello-world` も表示されているはずです。

In [ ]:
!docker images

これは Docker Hub のパブリック レジストリからプルされたイメージです。イメージ ID（Image ID）は `SHA256` ハッシュ形式であり、プロビジョニングされた Docker イメージを指定します。Docker デーモンはローカルでイメージが見つからない場合、デフォルトでパブリック レジストリからイメージを検索します。コンテナを再度実行してみましょう。

これで、`docker run hello-world` を再度実行しても、Artifact Registry からダウンロードする必要はありません。

実行中のすべての Docker コンテナを確認するには、`docker ps` を使用します。

In [ ]:
!docker ps

実行中のコンテナはありません。**注: Vertex AI Workbench（旧 AI Platform Notebooks）で実行している場合は、コンテナが 1 つ実行中として表示されます。**

先ほど実行した `hello-world` コンテナはすでに終了しています。実行が完了したコンテナを含め、すべてのコンテナを確認するには、`docker ps -a` を実行します。

In [ ]:
!docker ps -a

ここには、コンテナを識別するために Docker によって生成された UUID であるコンテナ ID（Container ID）や、実行に関するその他のメタデータが表示されます。コンテナ名（Names）もランダムに生成されますが、`docker run --name [container-name] hello-world` を使用して指定することもできます。

## Docker コンテナのビルド

シンプルな Node.js アプリケーションに基づく Docker イメージをビルドしてみましょう。

新しいテキスト ファイルを開き、次の内容を記述します。ファイルを `dockerfiles` というフォルダに保存し、ファイル名を `intro.docker` とします。

```bash
# 親イメージとして公式の Node ランタイムを使用
FROM node:6

# コンテナ内の作業ディレクトリを /app に設定
WORKDIR /app

# 現在のディレクトリの内容をコンテナ内の /app にコピー
ADD . /app

# コンテナのポート 80 を外部に公開
EXPOSE 80

# コンテナ起動時に node を使用して app.js を実行
CMD ["node", "./src/app.js"]
```

このファイルは、イメージのビルド方法を Docker デーモンに指示します。

最初の行はベースとなる親イメージを指定しており、この例では Node バージョン 6 の公式 Docker イメージです。
2 行目では、コンテナの作業（現在の）ディレクトリを設定します。
3 行目では、現在のディレクトリの内容（「.」で示される）をコンテナに追加します。
次に、コンテナのポートを公開してそのポートで接続を受け入れられるようにし、最後に node コマンドを実行してアプリケーションを起動します。

各行の動作を理解するには、その他の [Docker コマンド リファレンス](https://docs.docker.com/engine/reference/builder/#known-issues-run) を参照してください。

この Docker コンテナを使用して、シンプルな Node.js アプリを実行します。`app.js` を確認してみましょう。これはポート 80 でリッスンし、「Hello World」を返すシンプルな HTTP サーバーです。



それではイメージをビルドしましょう。「`.`」は現在のディレクトリを意味するため、Dockerfile のあるディレクトリ内からこのコマンドを実行する必要がある点に改めて注意してください。

`-t` は、`name:tag` 構文でイメージの名前とタグを指定するためのものです。イメージ名は `node-app` で、タグは `0.1` です。Docker イメージをビルドする際は、タグを指定することを強く推奨します。タグを指定しない場合、デフォルトで latest に設定され、新しいイメージと古いイメージを区別することが難しくなります。また、イメージのビルド時に、上記の Dockerfile の各行が中間コンテナ レイヤーとして生成される様子にも注目してください。

In [ ]:
!docker build -t node-app:0.1 -f dockerfiles/intro.docker .

イメージが正しく作成されたことを確認しましょう。

In [ ]:
!docker images

ほんの数秒前に作成された `node-app` リポジトリが表示されているはずです。

`node` がベースイメージであり、`node-app` がビルドしたイメージであることに注目してください。先に `node-app` を削除しないと、`node` を削除することはできません。イメージのサイズは VM と比較して比較的小さくなっています。`node:slim` や `node:alpine` など、node イメージの他のバージョンを使用すると、さらに小さいイメージを作成でき、移植性が向上します。コンテナ サイズの軽量化については、高度なトピックで詳しく説明します。すべてのバージョンは公式リポジトリで確認できます。

なお、Docker イメージからイメージを削除するには、`docker rmi [repository]:[tag]` を使用します。

## Docker コンテナの実行

次に、`docker run` コマンドを使用して、上でビルドしたイメージに基づいてコンテナを実行します。`--name` フラグを使用すると、必要に応じてコンテナに名前を付けることができます。また、`-p` はホストのポート 4000 をコンテナのポート 80 にマッピングするよう Docker に指示します。これにより、http://localhost:4000 でサーバーにアクセスできるようになります。ポート マッピングがないと、localhost でコンテナにアクセスできません。

In [ ]:
!docker ps -a

In [ ]:
!docker run -p 4000:80 --name my-app node-app:0.1

サーバーをテストするには、ターミナル ウィンドウを開いて次のコマンドを入力します。

```bash
curl http://localhost:4000
```

サーバーから `Hello World` という応答が返されるはずです。

コンテナは、最初のターミナルが実行されている間ずっと実行され続けます。コンテナを停止したい場合は、ターミナルで次のコマンドを実行してコンテナを停止および削除します。

```bash
docker stop my-app && docker rm my-app
```
しばらくするとコンテナが停止します。上のセルの実行が完了したことが確認できるはずです。

#### バックグラウンドでのコンテナの実行
コンテナを（ターミナルのセッションに関連付けずに）バックグラウンドで実行したい場合は、`-d` フラグを指定する必要があります。
次のコマンドを実行して、コンテナをバックグラウンドで起動します。

In [ ]:
!docker run -p 4000:80 --name my-app -d node-app:0.1

これでコンテナがバックグラウンドで実行されます。`docker ps` を使用して、実行中のコンテナのステータスを確認できます。

In [ ]:
!docker ps

docker ps の出力でコンテナが実行中になっていることを確認してください。`docker logs [container_id]` を実行すると、ログを確認できます。

In [ ]:
# 注: コンテナ ID は環境によって異なります
!docker logs b9d5fd6b8e33

以下のように表示されるはずです。
```bash
Server running at http://0.0.0.0:80/
```
コンテナの実行中にログ出力をリアルタイムで追跡したい場合は、`-f` オプションを使用します。

## 変更と公開

アプリケーションを変更して、Google Cloud Artifact Registry にプッシュしてみましょう。Artifact Registry リポジトリは、リポジトリのセットアップ プロセスですでに `asl-artifact-repo` として作成されています（詳細は **[セットアップ コード](../../../scripts/setup_on_jupyterlab.sh)** を参照してください）。

その後、ローカルのすべてのコンテナとイメージを削除してクリーンな環境をシミュレートし、Artifact Registry からコンテナをプルして実行します。これにより、Docker コンテナの移植性が実証されます。

### `app.js` の編集
テキスト エディタで `./src/app.js` ファイルを開き、「Hello World」を別の文字列に置き換えます。その後、この新しいイメージをビルドします。

In [ ]:
!docker build -t node-app:0.2 -f dockerfiles/intro.docker .

出力の `Step 2` で既存のキャッシュ レイヤーが使用されていることに注目してください。`Step 3` 以降は、`app.js` に変更を加えたため、レイヤーが変更されています。

新しいイメージ バージョンで別のコンテナを実行します。ホストのポート 80 ではなく 8000 をマッピングしている点に注目してください。ホスト ポート 4000 は既に使用されているため使用できません。

In [ ]:
!docker run -p 8000:80 --name my-app-2 -d node-app:0.2

 `docker ps` を使用して、両方のコンテナが実行中であることを確認できます。

In [ ]:
!docker ps

先ほどと同様に、`curl` を使用して両方のコンテナをテストしてみましょう。

In [ ]:
!curl http://localhost:8000

In [ ]:
!curl http://localhost:4000

なお、実行中のコンテナを停止するには、ターミナルで次のコマンドを実行するか、（バックグラウンドで実行されているため）このノートブックのセル内で実行できます。

### Artifact Registry への公開

ここでは、イメージを Google Cloud Artifact Registry にプッシュします。Artifact Registry がホストするプライベート レジストリにイメージをプッシュするには、レジストリ名を付けてイメージにタグを設定する必要があります。形式は `[hostname]/[project-id]/[repository name]/[image]:[tag]` です。

Artifact Registry の場合:

  * `[hostname]`= `<location>-docker.pkg.dev`。ここでは `us-docker.pkg.dev` を使用します。
  * `[project-id]`= ご自身のプロジェクト ID
  * `[repository name]`= リポジトリ名。ここでは `asl-artifact-repo` を使用します。
  * `[image]`= イメージ名
  * `[tag]`= 任意の文字列タグ。指定しない場合はデフォルトで `latest` になります。

In [ ]:
import os

PROJECT_ID = "your-gcp-project-here"  # ご自身のプロジェクト名（または ID）に置き換えてください
os.environ["PROJECT_ID"] = PROJECT_ID

`node-app:0.2` にタグを付けましょう。

In [ ]:
!docker images

In [ ]:
%%bash
docker tag node-app:0.2 us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2

Docker イメージを一覧表示すると、この新しくタグ付けされたリポジトリが表示されるはずです。

In [ ]:
!docker images

Artifact Registry にイメージをプッシュする前に、docker コマンドを認証する必要があります。

In [ ]:
%%bash
gcloud auth print-access-token | docker login -u oauth2accesstoken --password-stdin https://us-docker.pkg.dev

次に、このイメージを Artifact Registry にプッシュしましょう。

In [ ]:
%%bash
docker push us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2

Cloud Console のイメージ レジストリにアクセスして、イメージが Artifact Registry に存在することを確認します。コンソールで `ナビゲーション メニュー > Artifact Registry` に移動するか、下のセルの URL にアクセスしてください。

In [ ]:
%%bash
echo "http://us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app"

### 公開された Artifact Registry イメージのテスト

このイメージをテストしてみましょう。新しい VM を起動し、その VM に SSH 接続して gcloud をインストールすることもできます。ここでは簡単にするため、すべてのコンテナとイメージを削除してクリーンな環境をシミュレートします。

まず、`docker stop` と `docker rm` を使用してすべてのコンテナを停止および削除します。

In [ ]:
!docker stop my-app && docker rm my-app

In [ ]:
!docker stop my-app-2 && docker rm my-app-2

次に、`docker rmi` を使用して、上記で作成した Docker イメージを削除します。

In [ ]:
!docker images

In [ ]:
%%bash
docker rmi node-app:0.2
docker rmi us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2
docker rmi node-app:0.1
docker rmi node:6 
docker rmi -f hello-world:latest

`docker images` で、すべてのイメージが削除されたことを確認します。

In [ ]:
!docker images

この時点で、疑似的にクリーンな環境になっているはずです。次に、イメージをプルして実行します。

In [ ]:
%%bash
docker pull us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2
docker run -p 4000:80 -d us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2

先ほどと同様に、想定どおりに実行されていることを確認できます。

In [ ]:
!curl http://localhost:4000

Copyright 2020 Google LLC Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at https://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.